# MERA MultiQ: сравнение бейзлайнов и Qwen на 25 примерах

## 1. Настройки загрузки, установка библиотек и импорты

In [12]:
import torch

print("gpu доступен:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("видеокарта:", torch.cuda.get_device_name(0))

gpu доступен: True
видеокарта: Tesla T4


In [2]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [3]:
%pip -q install -U "transformers>=4.56.0" datasets accelerate pymorphy3 scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [8]:
import re
import random
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import torch
import pymorphy3
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    StoppingCriteria,
    StoppingCriteriaList,
)
from IPython.display import display
from tqdm.auto import tqdm

SEED = 42
N_EXAMPLES = 25

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("pytorch:", torch.__version__)
print("устройство:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

pytorch: 2.11.0+cu130
устройство: cuda
gpu: Tesla T4


## 2. Загрузка датасета и случайная выборка из 25 примеров

In [13]:
ds = load_dataset("MERA-evaluation/MERA", "multiq", split="train")

print("размер трейн:", len(ds))
print("поля датасета:", ds.column_names)
print("поля инпутов:", ds.features["inputs"])

sampled_ds = ds.shuffle(seed=SEED).select(range(N_EXAMPLES))
assert len(sampled_ds) == N_EXAMPLES

print(f"\nвыбрано {len(sampled_ds)} случайных примеров (seed={SEED}).")
print("поля первого примера:")
for key, value in sampled_ds[0].items():
    print(f"- {key}: {str(value)[:250]}")

размер трейн: 1056
поля датасета: ['instruction', 'inputs', 'outputs', 'meta']
поля инпутов: {'text': Value('string'), 'support_text': Value('string'), 'question': Value('string')}

выбрано 25 случайных примеров (seed=42).
поля первого примера:
- instruction: Найдите в приведенных двух текстах ответ на заданный вопрос и выпишите этот ответ. Он должен являться словом или короткой фразой, дополнительные объяснения приводить не нужно.
Текст 1: {support_text}
Текст 2: {text}
Вопрос: {question}
Ответ:
- inputs: {'text': 'Вятка ( , , ) — река в Европейской части России, крупнейший правый приток Камы (бассейн Волги) и второй по величине после Белой. Длина реки — 1314 км, площадь водосборного бассейна — 129 000 кв км.\\n\\nГеография.\\nСхема бассейна Вятки. На
- outputs: Каму;Камы
- meta: {'id': 129, 'bridge_answers': 'Вятки'}


## 3. Переменные в instruction и эталонные ответы

In [19]:
def build_prompt(example):
    inputs = example["inputs"]
    if not isinstance(inputs, dict):
        raise TypeError(f"ожидался dict в inputs, получено: {type(inputs).__name__}")
    try:
        return example["instruction"].format(**inputs)
    except KeyError as exc:
        raise KeyError(
            f"в instruction есть плейсхолдер {exc}, которого нет в inputs. "
            f"доступные поля: {list(inputs)}"
        ) from exc


def parse_references(value):
    if isinstance(value, (list, tuple, set)):
        refs = []
        for item in value:
            refs.extend(parse_references(item))
        return refs or [""]
    if value is None:
        return [""]
    parts = [part.strip() for part in str(value).split(";") if part.strip()]
    return parts or [""]


rows = []
for example in sampled_ds:
    rows.append({
        "id": example.get("meta", {}).get("id"),
        "question": example["inputs"].get("question", ""),
        "text": example["inputs"].get("text", ""),
        "support_text": example["inputs"].get("support_text", ""),
        "prompt": build_prompt(example),
        "reference_raw": example["outputs"],
        "references": parse_references(example["outputs"]),
        "bridge_answers": example.get("meta", {}).get("bridge_answers", ""),
    })

df = pd.DataFrame(rows)
assert len(df) == N_EXAMPLES
assert df["prompt"].map(bool).all()
assert df["references"].map(bool).all()

display(df[["id", "question", "reference_raw", "references"]].head(8))
print("\nпервый готовый промпт:\n")
print(df.loc[0, "prompt"])
print("\nэталоны:", df.loc[0, "references"])

,id,question,reference_raw,references
0,129,"В какую реку впадает река, притоком которой яв...",Каму;Камы,"[Каму, Камы]"
1,833,"Что за должность занимает глава государства, с...",президентов,[президентов]
2,243,Какое место является наивысшей точкой горного ...,Белуха;гора Белуха,"[Белуха, гора Белуха]"
3,584,"В какую реку впадает река, притоком которой яв...",Шаня,[Шаня]
4,548,"Как называется законодательный орган города, г...",ереванским городским советом,[ереванским городским советом]
5,807,"Как называется река, питающаяся от реки, прито...",Сабля,[Сабля]
6,47,"В какой лиге играет команда, тренера которой з...",Лиге МХ,[Лиге МХ]
7,218,"Как называется населенный пункт, в котором род...",Ереван,[Ереван]



первый готовый промпт:

Найдите в приведенных двух текстах ответ на заданный вопрос и выпишите этот ответ. Он должен являться словом или короткой фразой, дополнительные объяснения приводить не нужно.
Текст 1: Мурдюг (Большой Мурдюг) — река в России, протекает в Оричевском районе Кировской области. Устье реки находится в 494 км по левому берегу реки Вятки. Длина реки составляет 21 км. В 1,8 километрах от устья принимает слева реку Тумана.\n\nДанные водного реестра.\nПо данным государственного водного реестра России относится к Камскому бассейновому округу, водохозяйственный участок реки — Вятка от города Котельнич до водомерного поста посёлка городского типа Аркуль, речной подбассейн реки — Вятка. Речной бассейн реки — Кама.
Текст 2: Вятка ( , , ) — река в Европейской части России, крупнейший правый приток Камы (бассейн Волги) и второй по величине после Белой. Длина реки — 1314 км, площадь водосборного бассейна — 129 000 кв км.\n\nГеография.\nСхема бассейна Вятки. Начинается на Верхнек

## 4. Метрики: Exact Match и F1 по леммам

In [21]:
morph = pymorphy3.MorphAnalyzer()
_PUNCT_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)
_SPACE_RE = re.compile(r"\s+")

# нижний регистр, удаление пунктуации и схлопывание пробелов
def normalize_basic(text):
    text = str(text).casefold()
    text = _PUNCT_RE.sub(" ", text)
    text = text.replace("_", " ")
    return _SPACE_RE.sub(" ", text).strip()

def tokenize(text):
    return re.findall(r"[^\W_]+", normalize_basic(text), flags=re.UNICODE)


@lru_cache(maxsize=100_000)
def lemmatize_token(token):
    if re.search(r"[а-яё]", token, flags=re.IGNORECASE):
        return morph.parse(token)[0].normal_form
    return token


def answer_lemmas(text):
    return [lemmatize_token(token) for token in tokenize(text)]

In [23]:
def exact_match_one(prediction, reference):
    return float(normalize_basic(prediction) == normalize_basic(reference))


def exact_match(prediction, references):
    refs = parse_references(references)
    return max((exact_match_one(prediction, ref) for ref in refs), default=0.0)


def lemma_f1_one(prediction, reference):
    pred_lemmas = Counter(answer_lemmas(prediction))
    gold_lemmas = Counter(answer_lemmas(reference))
    pred_total = sum(pred_lemmas.values())
    gold_total = sum(gold_lemmas.values())

    if pred_total == 0 and gold_total == 0:
        return 1.0
    if pred_total == 0 or gold_total == 0:
        return 0.0

    common = sum((pred_lemmas & gold_lemmas).values())
    if common == 0:
        return 0.0

    precision = common / pred_total
    recall = common / gold_total
    return 2 * precision * recall / (precision + recall)

# максимальный F1 предсказания по любому допустимому эталону
def lemma_f1(prediction, references):
    refs = parse_references(references)
    return max((lemma_f1_one(prediction, ref) for ref in refs), default=0.0)

# средние метрики и построчные результаты
def evaluate_predictions(predictions, references=None):
    if references is None:
        references = df["references"].tolist()
    if len(predictions) != len(references):
        raise ValueError("число предсказаний не совпадает с числом эталонов")

    em_scores = [exact_match(p, g) for p, g in zip(predictions, references)]
    f1_scores = [lemma_f1(p, g) for p, g in zip(predictions, references)]
    summary = {
        "Exact Match": float(np.mean(em_scores)),
        "F1 (lemmas)": float(np.mean(f1_scores)),
    }
    details = pd.DataFrame({
        "prediction": predictions,
        "references": [" | ".join(parse_references(g)) for g in references],
        "exact_match": em_scores,
        "f1_lemmas": f1_scores,
    })
    return summary, details


# быстрая проверка нормализации и альтернативных эталонов
assert exact_match("КАМУ!", ["Каму", "Камы"]) == 1.0
assert lemma_f1("Каму", ["Каму", "Камы"]) == 1.0

## 5. Бейзлайны

Первый бейзлайн - это самый частотный биграмм из промпта, как в задании. В качестве второго я решила попробовать готовую русскоязычную модель RuBERT QA, которую дообучали на датасете SberQUAD. Она получает вопрос и тексты из примера, а затем выбирает из них фрагмент, который больше всего подходит на роль ответа.

На случайной выборке из 25 примеров эта модель показала exact match = 0.120 и F1 по леммам = 0.313. Это лучший результат среди проверенных мной бейзлайнов (в итоговую версию тетрадки они не вошли): модель справилась лучше, чем частотные биграммы, TF-IDF и подход с извлечением именованных сущностей.

In [30]:
from collections import Counter
import pandas as pd


def most_frequent_bigram(prompt):
    tokens = tokenize(str(prompt))
    bigrams = Counter(zip(tokens, tokens[1:]))

    if not bigrams:
        return tokens[0] if tokens else ""

    return " ".join(bigrams.most_common(1)[0][0])


# предсказания на выбранных 25 примерах
baseline_bigram = [
    most_frequent_bigram(prompt)
    for prompt in df["prompt"]
]

metrics_bigram, details_bigram = evaluate_predictions(baseline_bigram)

print("результаты простого бейзлайна:")
display(
    pd.DataFrame([{
        "Baseline": "Most frequent bigram",
        **metrics_bigram,
    }]).set_index("Baseline").style.format("{:.3f}")
)

результаты простого бейзлайна:


,Exact Match,F1 (lemmas)
Baseline,,
Most frequent bigram,0.040,0.040


In [32]:
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForQuestionAnswering


QA_MODEL_ID = "MilyaShams/rubert-russian-qa-sberquad"

tokenizer_qa = AutoTokenizer.from_pretrained(
    QA_MODEL_ID,
    use_fast=True,
)

model_qa = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_ID)
model_qa = model_qa.to(DEVICE)
model_qa.eval()


def qa_baseline(row, max_answer_tokens=30, stride=128):
    question = str(row["question"])

    context_parts = []
    for column in ["text", "support_text"]:
        value = row.get(column, "")
        if value is not None and str(value).strip() and str(value) != "nan":
            context_parts.append(str(value).strip())

    context = "\n\n".join(context_parts)
    if not context:
        return ""

    encoding = tokenizer_qa(
        question,
        context,
        max_length=512,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
        return_tensors="pt",
    )

    best_score = float("-inf")
    best_answer = ""

    with torch.no_grad():
        for i in range(encoding["input_ids"].shape[0]):
            inputs = {
                key: encoding[key][i:i + 1].to(DEVICE)
                for key in ["input_ids", "attention_mask", "token_type_ids"]
                if key in encoding
            }

            outputs = model_qa(**inputs)
            start_logits = outputs.start_logits[0].cpu().numpy()
            end_logits = outputs.end_logits[0].cpu().numpy()

            offsets = encoding["offset_mapping"][i].tolist()
            sequence_ids = encoding.sequence_ids(i)

            # учитываем только токены исходного контекста
            context_positions = [
                j for j, seq_id in enumerate(sequence_ids)
                if seq_id == 1 and offsets[j][1] > offsets[j][0]
            ]

            for start in context_positions:
                for end in context_positions:
                    if end < start:
                        continue
                    if end - start + 1 > max_answer_tokens:
                        break

                    char_start = offsets[start][0]
                    char_end = offsets[end][1]

                    if char_end <= char_start:
                        continue

                    answer = context[char_start:char_end].strip()
                    if not answer:
                        continue

                    score = float(start_logits[start] + end_logits[end])

                    if score > best_score:
                        best_score = score
                        best_answer = answer

    return best_answer


# ответы на той же выборке из 25 примеров
qa_predictions = [
    qa_baseline(row)
    for row in tqdm(
        df.to_dict("records"),
        total=len(df),
        desc="RuBERT QA",
    )
]
qa_metrics, qa_details = evaluate_predictions(qa_predictions)


# итоговое сравнение двух бейзлайнов
baseline_table = pd.DataFrame([
    {
        "Baseline": "Most frequent bigram",
        **metrics_bigram,
    },
    {
        "Baseline": "RuBERT QA (SberQUAD)",
        **qa_metrics,
    },
]).set_index("Baseline")

print("сравнение бейзлайнов:")
display(baseline_table.style.format("{:.3f}"))


# примеры предсказаний для проверки ошибок
display(pd.DataFrame({
    "question": df["question"],
    "gold": df["references"].map(lambda refs: " | ".join(refs)),
    "bigram": baseline_bigram,
    "RuBERT QA": qa_predictions,
    "EM": qa_details["exact_match"],
    "F1": qa_details["f1_lemmas"],
}).head(10))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RuBERT QA:   0%|          | 0/25 [00:00<?, ?it/s]

сравнение бейзлайнов:


,Exact Match,F1 (lemmas)
Baseline,,
Most frequent bigram,0.040,0.040
RuBERT QA (SberQUAD),0.120,0.313


,question,gold,bigram,RuBERT QA,EM,F1
0,"В какую реку впадает река, притоком которой яв...",Каму | Камы,река в,в Каму,0.0,0.666667
1,"Что за должность занимает глава государства, с...",президентов,3 февраля,российский военный лётчик,0.0,0.000000
2,Какое место является наивысшей точкой горного ...,Белуха | гора Белуха,востоке с,гора Белуха,1.0,1.000000
3,"В какую реку впадает река, притоком которой яв...",Шаня,река в,Суходрев,0.0,0.000000
4,"Как называется законодательный орган города, г...",ереванским городским советом,гор шаваршевич,ереванским городским советом,1.0,1.000000
5,"Как называется река, питающаяся от реки, прито...",Сабля,река в,Кисьма,0.0,0.000000
6,"В какой лиге играет команда, тренера которой з...",Лиге МХ,лос кобос,в Лиге МХ,0.0,0.800000
7,"Как называется населенный пункт, в котором род...",Ереван,армянский футболист,в Ереване,0.0,0.666667
8,"Какая должность у главы государства в стране, ...",президентов,мухаммед ашави,"сирийский государственный деятель, министр ино...",0.0,0.000000
9,"Кто является главой администрации в городе, гд...",Евгению Тефтелеву,диана александровна,Борис Дубровский,0.0,0.000000


## 6. Модель Qwen/Qwen3-0.6B, chat template и генерация без thinking

In [35]:
# генерация на cpu может быть медленной, для colab я выбрала gpu до запуска этой ячейки

from transformers import __version__ as transformers_version
print("Transformers:", transformers_version)

MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

if DEVICE.type in {"cuda", "mps"}:
    MODEL_DTYPE = torch.float16
else:
    MODEL_DTYPE = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
model.to(DEVICE)
model.eval()

print("модель загружена:", MODEL_ID)

Transformers: 5.21.0


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

модель загружена: Qwen/Qwen3-0.6B


In [36]:
def make_chat_prompt(prompt, system_prompt=None, enable_thinking=False):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )


@torch.inference_mode()
def generate_without_thinking(prompts, system_prompt=None, max_new_tokens=96, batch_size=None):
    chat_prompts = [
        make_chat_prompt(prompt, system_prompt=system_prompt, enable_thinking=False)
        for prompt in prompts
    ]
    if batch_size is None:
        batch_size = 2 if DEVICE.type == "cuda" else 1

    predictions = []
    for start in tqdm(range(0, len(chat_prompts), batch_size), desc="генерация", unit="batch"):
        batch_texts = chat_prompts[start:start + batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=False)
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        input_length = inputs["input_ids"].shape[1]
        new_tokens = generated[:, input_length:]
        predictions.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))

    return [prediction.strip() for prediction in predictions]

# считает метрики и сохраняет предсказания в общей таблице примеров
def evaluate_and_save(name, predictions):
    df[name] = predictions
    metrics, details = evaluate_predictions(predictions)
    df[f"em_{name}"] = details["exact_match"].to_numpy()
    df[f"f1_{name}"] = details["f1_lemmas"].to_numpy()
    print(f"{name}: EM={metrics['Exact Match']:.3f}; F1 по леммам={metrics['F1 (lemmas)']:.3f}")
    return metrics, details


PROMPTS = df["prompt"].tolist()
REFERENCES = df["references"].tolist()
MAX_NEW_TOKENS = 96
BATCH_SIZE = 2 if DEVICE.type == "cuda" else 1

# диагностируем chat template на одном промпте, не запуская генерацию
print(make_chat_prompt(PROMPTS[0], enable_thinking=False)[-500:])

й части России, крупнейший правый приток Камы (бассейн Волги) и второй по величине после Белой. Длина реки — 1314 км, площадь водосборного бассейна — 129 000 кв км.\n\nГеография.\nСхема бассейна Вятки. Начинается на Верхнекамской возвышенности на севере Удмуртии у ж.-д. разъезда Перелом. Впадает в Каму в 17 км ниже города Мамадыш. Высота истока — 240 м над уровнем моря.
Вопрос: В какую реку впадает река, притоком которой является Мурдюг?
Ответ:<|im_end|>
<|im_start|>assistant
<think>

</think>




## 7. Эксперимент 1 - без thinking и без system prompt

In [37]:
predictions_exp1 = generate_without_thinking(
    PROMPTS,
    system_prompt=None,
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=BATCH_SIZE,
)
metrics_exp1, details_exp1 = evaluate_and_save("prediction_exp1", predictions_exp1)

display(pd.DataFrame([metrics_exp1], index=["Qwen3-0.6B / без system / без thinking"]).style.format("{:.3f}"))
display(details_exp1.head(10))

генерация:   0%|          | 0/13 [00:00<?, ?batch/s]

prediction_exp1: EM=0.080; F1 по леммам=0.238


,Exact Match,F1 (lemmas)
Qwen3-0.6B / без system / без thinking,0.080,0.238


,prediction,references,exact_match,f1_lemmas
0,"В реку впадает река, притоком которой является...",Каму | Камы,0.0,0.0
1,Герой Российской Федерации.,президентов,0.0,0.0
2,"Наивысшая точка горного хребта, к которому при...",Белуха | гора Белуха,0.0,0.0
3,Суходревка.,Шаня,0.0,0.0
4,Ереван,ереванским городским советом,0.0,0.0
5,Ворговаш.,Сабля,0.0,0.0
6,Карлос де лос Кобос играет в лиге **Керетаро**.,Лиге МХ,0.0,0.2
7,Ереван,Ереван,1.0,1.0
8,Президент,президентов,0.0,1.0
9,Диана Александровна Циринг.,Евгению Тефтелеву,0.0,0.0


## 8. Эксперимент 2 - с system prompt, без thinking

Параметры генерации оставляю такими же, как в первом эксперименте.

In [38]:
SYSTEM_PROMPT = """ты отвечаешь на вопрос, опираясь на два предоставленных текста. внимательно сопоставь информацию из текстов: ответ может требовать связать факты из них. не используй внешние знания и не добавляй сведения, которых нет в текстах. сформулируй краткий точный ответ, по возможности близкий к формулировке в тексте. выведи только ответ, без объяснений и дополнительных комментариев."""

predictions_exp2 = generate_without_thinking(
    PROMPTS,
    system_prompt=SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,  # те же параметры, что в эксперименте 1
    batch_size=BATCH_SIZE,
)
metrics_exp2, details_exp2 = evaluate_and_save("prediction_exp2", predictions_exp2)

display(pd.DataFrame([metrics_exp2], index=["Qwen3-0.6B / system / без thinking"]).style.format("{:.3f}"))
display(details_exp2.head(10))

генерация:   0%|          | 0/13 [00:00<?, ?batch/s]

prediction_exp2: EM=0.080; F1 по леммам=0.227


,Exact Match,F1 (lemmas)
Qwen3-0.6B / system / без thinking,0.080,0.227


,prediction,references,exact_match,f1_lemmas
0,Мурдюг впадает в Каму.,Каму | Камы,0.0,0.400000
1,"Глава государства, ставший местом смерти Роман...",президентов,0.0,0.000000
2,наивысшая точка горного хребта — гора Белуха (...,Белуха | гора Белуха,0.0,0.400000
3,Суходревка.,Шаня,0.0,0.000000
4,Ереван,ереванским городским советом,0.0,0.000000
5,Ворговаш.,Сабля,0.0,0.000000
6,"Команда, тренер которой зовут Карлос де лос Ко...",Лиге МХ,0.0,0.266667
7,Ереван,Ереван,1.0,1.000000
8,Министр иностранных дел,президентов,0.0,0.000000
9,Ответ: Евгений Тефтелев,Евгению Тефтелеву,0.0,0.400000


In [41]:
SYSTEM_PROMPT = """ты отвечаешь на вопрос, используя два предоставленных текста, внимательно прочитай оба текста и сопоставь содержащиеся в них факты. ответ может требовать связать информацию из разных текстов: например, найти объект в одном тексте, установить связанную с ним характеристику в другом и на основе этого определить ответ. не ограничивайся поиском слов, которые напрямую совпадают с вопросом. используй только информацию из предоставленных текстов. Не добавляй внешние знания и не делай выводов, которые не подтверждаются текстами. Перед ответом проверь, что выбранная информация действительно отвечает на вопрос, а не просто упоминается в контексте. дай краткий и точный ответ, по возможности используя формулировку из текста. не добавляй пояснений, вводных слов, лишних определений или предлогов, если они не нужны для ответа. если спрашивается имя, название, дата или должность, укажи именно их, без дополнительного описания. выведи только ответ"""

predictions_exp3 = generate_without_thinking(
    PROMPTS,
    system_prompt=SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=BATCH_SIZE,
)
metrics_exp3, details_exp3 = evaluate_and_save("prediction_exp3", predictions_exp3)

display(pd.DataFrame([metrics_exp3], index=["Qwen3-0.6B / system / без thinking"]).style.format("{:.3f}"))
display(details_exp3.head(10))

генерация:   0%|          | 0/13 [00:00<?, ?batch/s]

prediction_exp3: EM=0.040; F1 по леммам=0.341


,Exact Match,F1 (lemmas)
Qwen3-0.6B / system / без thinking,0.040,0.341


,prediction,references,exact_match,f1_lemmas
0,Ответ: В Каму.,Каму | Камы,0.0,0.500000
1,"Глава государства, ставшего местом смерти Рома...",президентов,0.0,0.181818
2,Гора Белуха.,Белуха | гора Белуха,1.0,1.000000
3,Суходревка впадает в реку Суходрев.,Шаня,0.0,0.000000
4,Ереванский городской совет.,ереванским городским советом,0.0,1.000000
5,Ворговаш.,Сабля,0.0,0.000000
6,"Карлос де лос Кобос играет в лиге ""Керетаро"".",Лиге МХ,0.0,0.200000
7,Ответ: Ереван.,Ереван,0.0,0.666667
8,Ответ: министр иностранных дел.,президентов,0.0,0.000000
9,Ответ: Евгений Тефтелев.,Евгению Тефтелеву,0.0,0.400000


In [42]:
SYSTEM_PROMPT = """ответь на вопрос, сопоставив факты из обоих текстов. используй только тексты. выведи краткий ответ без пояснений"""

predictions_exp4 = generate_without_thinking(
    PROMPTS,
    system_prompt=SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=BATCH_SIZE,
)
metrics_exp4, details_exp4 = evaluate_and_save("prediction_exp4", predictions_exp4)

display(pd.DataFrame([metrics_exp4], index=["Qwen3-0.6B / system / без thinking"]).style.format("{:.3f}"))
display(details_exp4.head(10))

генерация:   0%|          | 0/13 [00:00<?, ?batch/s]

prediction_exp4: EM=0.000; F1 по леммам=0.201


,Exact Match,F1 (lemmas)
Qwen3-0.6B / system / без thinking,0.000,0.201


,prediction,references,exact_match,f1_lemmas
0,Мурдюг впадает в Каму.,Каму | Камы,0.0,0.400000
1,"Глава государства, ставший местом смерти Роман...",президентов,0.0,0.181818
2,"Наивысшая точка горного хребта, к которому при...",Белуха | гора Белуха,0.0,0.285714
3,Суходревка.,Шаня,0.0,0.000000
4,Ереван.,ереванским городским советом,0.0,0.000000
5,Ворговаш.,Сабля,0.0,0.000000
6,"Карлос де лос Кобос играет в лиге ""Керетаро"".",Лиге МХ,0.0,0.200000
7,Ответ: Ереван.,Ереван,0.0,0.666667
8,Министр иностранных дел Сирии.,президентов,0.0,0.000000
9,Ответ: Евгений Тефтелев.,Евгению Тефтелеву,0.0,0.400000


### Сравнение системных промптов

Я сравнила три варианта системного промпта для Qwen3-0.6B без thinking. Во всех экспериментах использовались одинаковые параметры генерации и выборка из 25 примеров.

Первый промпт содержал общие инструкции: опираться на оба текста, сопоставлять информацию, не использовать внешние знания и давать краткий ответ. Он показал exact match = 0.080 и F1 по леммам = 0.227.

Во втором варианте я подробно описала, как нужно связывать факты из разных текстов, проверять соответствие ответа вопросу и избегать лишних слов. Exact match снизился до 0.040, зато F1 вырос до 0.3416, и это лучший результат среди трёх вариантов по данной метрике. Например, модель смогла правильно определить наивысшую точку горного хребта и законодательный орган Еревана, хотя во втором случае ответ отличался от эталона по форме слова.

Третий вариант содержал короткую инструкцию, результат оказался самым низким: exact match = 0.000 и F1 = 0.201.

На этой выборке более подробный промпт помог увеличить совпадение ответов с эталонами на уровне отдельных слов, но не улучшил exact match. Короткая инструкция, наоборот, дала худший результат по обеим метрикам. При этом все три варианта показали среднее качество: даже подробная инструкция не гарантирует, что модель даст правильный ответ. Наша выборка состоит всего из 25 примеров, поэтому эти результаты предварительные.

## 8. Эксперимент 3 — thinking с ограничением токенов в два этапа

Бюджет генерации на один пример - THINKING_TOKENS + ANSWER_TOKENS. Сначала нужно сгенерировать максимум THINKING_TOKENS и остановиться на специальном токене </think>. Если модель не успела закрыть блок, надо дописать этот специальный токен вручную. Затем продолжаем уже накопленную последовательность с отдельным лимитом на финальный ответ. В метрики передаётся только финальный ответ (без thinking).

Используем тот же SYSTEM_PROMPT, что и в эксперименте 2. Оставим do_sample=False в третьем эксперименте для воспроизводимости. Изменяется режим thinking и появляется отдельный бюджет на рассуждение.

In [45]:
THINKING_TOKENS = 256
ANSWER_TOKENS = 96
TOTAL_NEW_TOKEN_BUDGET = THINKING_TOKENS + ANSWER_TOKENS

think_end_id = tokenizer.convert_tokens_to_ids("</think>")
if think_end_id is None or think_end_id == tokenizer.unk_token_id:
    raise ValueError("токен </think> не найден в словаре токенизатора выбранной модели")
print(f"лимит этапа thinking: {THINKING_TOKENS}; лимит ответа: {ANSWER_TOKENS}; суммарный бюджет: {TOTAL_NEW_TOKEN_BUDGET} токенов.")

# останавливает генерацию, когда последний новый токен - заданный special token
class StopOnTokenId(StoppingCriteria):
    def __init__(self, token_id):
        self.token_id = token_id

    # в этом эксперименте генерация идёт по одному примеру
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[:, -1].eq(self.token_id)


@torch.inference_mode()
# генерирует thinking до </think> (или лимита), продолжает финальным ответом
def generate_with_limited_thinking(prompt, system_prompt=None,
                                   thinking_tokens=THINKING_TOKENS,
                                   answer_tokens=ANSWER_TOKENS):
    chat_text = make_chat_prompt(
        prompt,
        system_prompt=system_prompt,
        enable_thinking=True,
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(DEVICE)
    input_ids = inputs["input_ids"]
    attention_mask = inputs.get("attention_mask", torch.ones_like(input_ids))

    # этап 1: отдельный лимит на thinking. StoppingCriteria следит за ID токена </think>
    thinking_generation = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=thinking_tokens,
        do_sample=False,
        use_cache=True,
        stopping_criteria=StoppingCriteriaList([StopOnTokenId(think_end_id)]),
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated_thinking = thinking_generation[0, input_ids.shape[1]:]

    end_positions = (generated_thinking == think_end_id).nonzero(as_tuple=True)[0]
    closed_naturally = end_positions.numel() > 0

    # до </think> — текст рассуждения; сам маркер сохраняем для продолжения
    if closed_naturally:
        end_position = int(end_positions[0].item())
        thinking_only_ids = generated_thinking[:end_position]
        continuation_ids = generated_thinking[:end_position + 1]

    # если модель остановилась по лимиту или EOS, удаляем EOS и закрываем блок вручную
    else:
        thinking_only_ids = generated_thinking
        if tokenizer.eos_token_id is not None:
            eos_positions = (thinking_only_ids == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
            if eos_positions.numel() > 0:
                thinking_only_ids = thinking_only_ids[:int(eos_positions[0].item())]
        closing_id = torch.tensor([think_end_id], device=input_ids.device, dtype=input_ids.dtype)
        continuation_ids = torch.cat([thinking_only_ids, closing_id])

    # этап 2: продолжаем весь накопленный контекст после закрытого thinking-блока
    continuation_input = torch.cat([input_ids, continuation_ids.unsqueeze(0)], dim=1)
    continuation_mask = torch.ones_like(continuation_input)
    answer_generation = model.generate(
        input_ids=continuation_input,
        attention_mask=continuation_mask,
        max_new_tokens=answer_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    answer_ids = answer_generation[0, continuation_input.shape[1]:]

    return {
        "thinking": tokenizer.decode(thinking_only_ids, skip_special_tokens=True).strip(),
        "answer": tokenizer.decode(answer_ids, skip_special_tokens=True).strip(),
        "thinking_closed_naturally": bool(closed_naturally),
        "thinking_tokens": int(thinking_only_ids.numel()),
        "answer_tokens": int(answer_ids.numel()),
    }


results_exp3 = []
for i, prompt in enumerate(tqdm(PROMPTS, desc="thinking: два этапа", unit="пример"), start=1):
    result = generate_with_limited_thinking(prompt, system_prompt=SYSTEM_PROMPT)
    results_exp3.append(result)
    print(
        f"{i:02d}/{len(PROMPTS)}   "
        f"thinking={result['thinking_tokens']} токенов, "
        f"answer={result['answer_tokens']} токенов, "
        f"</think> закрыт моделью: {result['thinking_closed_naturally']}"
    )

predictions_exp3 = [result["answer"] for result in results_exp3]
metrics_exp3, details_exp3 = evaluate_and_save("prediction_exp3", predictions_exp3)
display(pd.DataFrame([metrics_exp3], index=["Qwen3-0.6B / system / thinking"]).style.format("{:.3f}"))
display(details_exp3.head(10))

лимит этапа thinking: 256; лимит ответа: 96; суммарный бюджет: 352 токенов.


thinking: два этапа:   0%|          | 0/25 [00:00<?, ?пример/s]

01/25   thinking=256 токенов, answer=9 токенов, </think> закрыт моделью: False
02/25   thinking=256 токенов, answer=35 токенов, </think> закрыт моделью: False
03/25   thinking=32 токенов, answer=32 токенов, </think> закрыт моделью: True
04/25   thinking=256 токенов, answer=8 токенов, </think> закрыт моделью: False
05/25   thinking=251 токенов, answer=5 токенов, </think> закрыт моделью: True
06/25   thinking=256 токенов, answer=8 токенов, </think> закрыт моделью: False
07/25   thinking=62 токенов, answer=8 токенов, </think> закрыт моделью: True
08/25   thinking=210 токенов, answer=6 токенов, </think> закрыт моделью: True
09/25   thinking=203 токенов, answer=13 токенов, </think> закрыт моделью: True
10/25   thinking=256 токенов, answer=14 токенов, </think> закрыт моделью: False
11/25   thinking=183 токенов, answer=9 токенов, </think> закрыт моделью: True
12/25   thinking=220 токенов, answer=8 токенов, </think> закрыт моделью: True
13/25   thinking=162 токенов, answer=8 токенов, </think> 

,Exact Match,F1 (lemmas)
Qwen3-0.6B / system / thinking,0.120,0.324


,prediction,references,exact_match,f1_lemmas
0,Ответ: Вятка.,Каму | Камы,0.0,0.0
1,"Глава государства, ставший местом смерти Роман...",президентов,0.0,0.0
2,"Холзун — наивысшая точка горного хребта, к кот...",Белуха | гора Белуха,0.0,0.0
3,Суходревка.,Шаня,0.0,0.0
4,городской совет,ереванским городским советом,0.0,0.8
5,Ворговаш.,Сабля,0.0,0.0
6,Керетаро.,Лиге МХ,0.0,0.0
7,Ереван.,Ереван,1.0,1.0
8,Министр иностранных дел.,президентов,0.0,0.0
9,Ответ: Евгений Тефтелев.,Евгению Тефтелеву,0.0,0.4


## 9. Итоги

Все методы оценивались на одной случайной выборке из 25 примеров MERA MultiQ с помощью exact match и F1 по леммам.

- Лучшие результаты по exact match показали RuBERT QA (SberQUAD) и Qwen3-0.6B с thinking — 0.120 у обеих моделей, то есть по 3 точных совпадения из 25.
- Самый высокий F1 получился у Qwen3-0.6B с подробным системным промптом - 0.341. При этом exact match = 0.040, поэтому частичное совпадение слов с эталоном не всегда означало правильный ответ целиком.
- Режим thinking улучшил результаты Qwen3-0.6B по сравнению с генерацией без системного промпта: exact match вырос с 0.080 до 0.120, а F1 - с 0.227 до 0.324. В 12 из 25 примеров был исчерпан лимит токенов на рассуждения.
- Простые бейзлайны оказались слабыми: частотные биграммы дали exact match = 0.040, а TF-IDF и извлечение именованных сущностей практически не справились с задачей.
- Ошибки моделей показывают, что основной проблемой остаются вопросы, требующие связать несколько фактов из разных текстов, а также точное формулирование ответа.